In [ ]:
import pandas as pd
import numpy as np
from pymongo import MongoClient, DESCENDING
from sklearn.ensemble import IsolationForest
from prophet import Prophet
import matplotlib.pyplot as plt

In [ ]:
# -------- 1. CONNECT MONGO --------
client = MongoClient("string") 
db = client["IOTDeviceMonitor"]
collection = db["FUTU00_DataMonitor"]  

# DUMMY
# client = MongoClient("mongodb://localhost:27017/") 
# db = client["IOT"]
# collection = db["NEVCOE_DataMonitor_Dummy"]  


In [ ]:
# -------- 2. FETCH DATA --------
query = {"EnergymeterId": "FUTU0000000004000002"}
# A1 -> Energy

cursor = collection.find(query).sort("_id", DESCENDING)
data = list(cursor)

In [ ]:
# -------- 3. TRANSFORM --------
df = pd.DataFrame(data)

df["RTC"] = pd.to_datetime(df["RTC"])
df["A1"] = df["A1"].astype(float)

df = df.sort_values("RTC")

# Core signal
df["delta_volume"] = df["A1"].diff().fillna(0)

# Remove negative resets (important)
df = df[df["delta_volume"] >= 0]

print(df.head())

In [ ]:
# -------- 4. FEATURE ENGINEERING --------
df["hour"] = df["RTC"].dt.hour
df["minute"] = df["RTC"].dt.minute

df["rolling_mean"] = df["delta_volume"].rolling(5).mean().fillna(0)
df["rolling_std"] = df["delta_volume"].rolling(5).std().fillna(0)
print(df.head())

In [ ]:
# -------- 5. ANOMALY DETECTION --------
model = IsolationForest(contamination=0.02, random_state=42)
df["anomaly"] = model.fit_predict(df[["delta_volume"]])
df["anomaly"] = df["anomaly"].map({1: 0, -1: 1})

# -------- 6. CLEAN DATA FOR FORECAST --------
clean_df = df[df["anomaly"] == 0]

prophet_df = clean_df[["RTC", "delta_volume"]].rename(
    columns={"RTC": "ds", "delta_volume": "y"}
)

In [ ]:
# -------- 7. FORECAST --------
model_p = Prophet()
model_p.fit(prophet_df)

future = model_p.make_future_dataframe(periods=10, freq="min")
forecast = model_p.predict(future)

In [ ]:
# -------- 8. PLOTTING --------
plt.figure(figsize=(12, 6))

# Actual data
plt.plot(df["RTC"], df["delta_volume"], label="Actual", marker="o")

# Anomalies
anomalies = df[df["anomaly"] == 1]
plt.scatter(anomalies["RTC"], anomalies["delta_volume"], label="Anomaly")

# Forecast
plt.plot(forecast["ds"], forecast["yhat"], linestyle="--", label="Forecast")

# Confidence interval
plt.fill_between(
    forecast["ds"],
    forecast["yhat_lower"],
    forecast["yhat_upper"],
    alpha=0.2
)

plt.title("Flow Consumption Forecast & Anomaly Detection")
plt.xlabel("Time")
plt.ylabel("Delta Volume")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# -------- 9. OUTPUT --------
print("\n=== Latest Processed Data ===")
# print(df)

print("\n=== Forecast ===")
print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(10))